# Python Function operator in rocAL
This example demonstrates how to use `fn.python_function` to inject custom Python processing into a rocAL pipeline. We build a simple image pipeline and apply Python-based brightness, crop, and horizontal flip steps.

<font size="12"> Common Code </font>

In [ ]:
import numpy as np
from functools import partial
import random
import os
import matplotlib.pyplot as plt
%matplotlib inline

import amd.rocal.fn as fn
import amd.rocal.types as types
from amd.rocal.pipeline import Pipeline
from amd.rocal.plugin.generic import ROCALClassificationIterator


<font size= "12" >Configuring rocAL pipeline </font>

<div class="alert alert-block alert-warning">
<b>Note:</b> Set the ROCAL_DATA_PATH environment variable before running the notebook.
</div>


In [ ]:
# Check if ROCAL_DATA_PATH is set
rocal_data_path = os.environ.get('ROCAL_DATA_PATH')
if rocal_data_path is None:
    raise EnvironmentError("ROCAL_DATA_PATH environment variable is not set. Please set it to the correct path.")
else:
    print(f"ROCAL_DATA_PATH IS SET TO: {rocal_data_path}")
image_dir = os.path.join(rocal_data_path, 'rocal_data', 'images_jpg', 'labels_folder')

# Pipeline configuration
rocal_cpu = True  # python_function is supported on Host backend
batch_size = 1
num_threads = 4
device_id = 0
seed = 2


<font size="12">Defining Python functions </font>

In [ ]:
def random_augmentation(probability, augmented, original):
    condition = random.random() < probability
    neg_condition = condition ^ True
    return condition * augmented + neg_condition * original

def brightness_fn(img):
    # Apply random brightness scaling in [0.7, 1.3]
    brightness_scale = random_augmentation(0.5, random.uniform(0.7, 1.3), 1.0)
    return (img * brightness_scale).astype(np.uint8)  # Ensure uint8 output

def crop_fn(img, crop_size):
    # Crop along height and width (NHWC layout)
    return img[:, :crop_size[0], :crop_size[1], :]

def flip_fn(img):
    # Horizontal flip with 50% probability
    if random.random() < 0.5:
        return img[:, :, ::-1, :]
    else:
        return img

class NormalizeWithStats:
    def __init__(self, mean, std):
        self.mean = np.array(mean).reshape(1, 1, 1, -1)
        self.std = np.array(std).reshape(1, 1, 1, -1)
    def __call__(self, batch):
        # Normalize using user-provided mean and std
        return ((batch - self.mean) / self.std).astype(np.float32)

# Bind crop size via partial for convenience
crop_image_fn = partial(crop_fn, crop_size=(224, 224))
normalizer = NormalizeWithStats(mean=[0.485 * 255, 0.456 * 255, 0.406 * 255],
                                std=[0.229 * 255, 0.224 * 255, 0.225 * 255])


<font size="12">Python function pipeline </font>

We read images using the file reader and decode them. Then we chain `fn.python_function` calls to apply random brightness, crop to 224×224, and an optional horizontal flip. Finally, we demonstrate passing a callable class to normalize the batch.

In [ ]:
pipe = Pipeline(batch_size=batch_size, num_threads=num_threads, device_id=device_id, seed=seed, rocal_cpu=rocal_cpu, tensor_layout=types.NHWC)
with pipe:
    jpegs, _ = fn.readers.file(file_root=image_dir)
    images = fn.decoders.image(jpegs, file_root=image_dir, output_type=types.RGB, shard_id=0, num_shards=1, random_shuffle=False)
    rand_brightness = fn.python_function(images, function=brightness_fn, dtype=types.UINT8, layout=types.NHWC)
    cropped = fn.python_function(rand_brightness, function=crop_image_fn, output_dims=(224, 224, 3), dtype=types.UINT8, layout=types.NHWC)
    flipped = fn.python_function(cropped, function=flip_fn, dtype=types.UINT8, layout=types.NHWC)
    normalized = fn.python_function(flipped, function=normalizer, dtype=types.FLOAT, layout=types.NHWC)
    pipe.set_outputs(flipped, normalized)  # Return both for visualization

pipe.build()
data_loader = ROCALClassificationIterator(pipe)

<font size ="12">Visualizing outputs</font>

We display both the flipped (uint8) output and the normalized (float) output for the same sample. For normalized data, we clip to [0, 255] for visualization.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
# Iterate one batch and display the first sample of both outputs
for i, it in enumerate(data_loader, 0):
    flipped_img = it[0][0][0].astype(np.uint8)  # [images][flipped][sample 0]
    axes[0].imshow(flipped_img)
    axes[0].set_title('flipped')

    normalized_img = it[0][1][0]  # [images][normalized][sample 0]
    normalized_img_vis = np.clip(normalized_img*255, 0, 255).astype(np.uint8)
    axes[1].imshow(normalized_img_vis)
    axes[1].set_title('normalized (clipped)')
    break
data_loader.reset()
plt.tight_layout()
plt.show()